# keigo-slm — train, evaluate & demo (Colab)

Fine-tunes Qwen3-1.7B to preserve Japanese politeness register in JP→EN translation, then
evaluates it against the golden set and lets you try it live.

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Add secrets via the 🔑 panel:
`HF_TOKEN` (required — to push the model) and `TEACHER_API_KEY` (for the LLM-judged eval).
Never paste keys inline — the cells read them from the 🔑 secrets. Run top to bottom.

In [ ]:
# 1. Install
!pip install -q unsloth "trl<0.10" peft datasets

In [ ]:
# 2. Get the code + log in to Hugging Face
!git clone -q https://github.com/akabraham06/keigo-slm.git
%cd keigo-slm
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login; login(os.environ['HF_TOKEN'])

In [ ]:
# 3. Build the balanced training set (real data: BSD business + Tatoeba casual).
#    Writes data/train_full.jsonl (~2,260 rows, 904/904/452). sft.py auto-uses it.
!PYTHONPATH=src python -m data.harvest_parallel --source bsd     --judge heuristic
!PYTHONPATH=src python -m data.harvest_parallel --source tatoeba --judge heuristic --max-per-band 4000
!PYTHONPATH=src python -m data.build_training_set --ratio 2 2 1

In [ ]:
# 4. Train (QLoRA SFT) on the balanced set → push as v2.
#    sft.py auto-detects data/train_full.jsonl. Drop --push-repo to train without publishing.
!PYTHONPATH=src python -m train.sft --model unsloth/Qwen3-1.7B --epochs 3 \
    --push-repo akabraham06/keigo-slm-qwen3-1.7b-v2

In [ ]:
# 5. Evaluate — base vs tuned vs frontier (naive + well-prompted), one table.
#    Key is read from the 🔑 secret TEACHER_API_KEY (never hardcode it).
#    Free alternative (no key): drop the two env lines + openai, and use  --no-llm --judge heuristic
import os
from google.colab import userdata
os.environ["TEACHER_API_KEY"]  = userdata.get("TEACHER_API_KEY")
os.environ["TEACHER_MODEL"]    = "claude-group/claude-sonnet-4-6"
os.environ["TEACHER_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
!pip install -q openai

!PYTHONPATH=src python -m eval.run_eval \
    --tuned akabraham06/keigo-slm-qwen3-1.7b-v2 \
    --base  unsloth/Qwen3-1.7B \
    --llm --llm-prompt llm_naive.md llm_baseline.md --judge llm

print("\n===== RESULTS TABLE =====")
print(open("results/results_table.md").read())

## 5b. Export the measured submission evidence

Run this immediately after evaluation, then copy the downloaded zip back into the repository.

In [ ]:
# Package the exact files used by the results table, public site, and error analysis.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

evidence = [
    Path("results/results_table.md"),
    Path("results/predictions.jsonl"),
    Path("results/error_analysis.md"),
    Path("results/eval_results.json"),
    Path("app/web/eval_results.json"),
]
with ZipFile("keigo-slm-evidence.zip", "w", ZIP_DEFLATED) as archive:
    for path in evidence:
        archive.write(path)
files.download("keigo-slm-evidence.zip")

## 6. Try it live

Load the models once, then translate any Japanese you type and see, per model, the output's
register and whether it **preserved** or **flattened** the source. The three-register examples
(casual → polite → formal) are the clearest thing to show a non-Japanese audience.

In [ ]:
# Live playground — compare base vs tuned on your own examples
import sys; sys.path.insert(0, "src")
from inference.playground import Playground

pg = Playground(base="unsloth/Qwen3-1.7B", tuned="akabraham06/keigo-slm-qwen3-1.7b-v2")

for jp in ["資料を拝見いたします。", "資料見とくね。", "コーヒーを飲みますか。"]:
    pg.compare(jp)

# pg.compare("会議は何時からですか。")   # ← try your own sentence
# pg.widget()                             # ← text-box + button UI